In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.utils.class_weight import compute_class_weight

sns.set_style("whitegrid")  # Plot style

# 1️⃣ Load Dataset
dataset_url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ["Pregnancies","Glucose","BloodPressure","SkinThickness","Insulin",
           "BMI","DiabetesPedigreeFunction","Age","Outcome"]
df = pd.read_csv(dataset_url, names=columns)
df.head()

# Features / Target
X = df.drop("Outcome", axis=1)
y = df["Outcome"]

# 2️⃣ ตรวจสอบความไม่สมดุลของ Class
print("Original class distribution:", Counter(y))
sns.countplot(x=y, palette="coolwarm")
plt.title("Class Distribution in Dataset")
plt.show()

# Split Train/Test (stratify เพื่อคงสัดส่วน class)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train class distribution:", Counter(y_train))
print("Test class distribution:", Counter(y_test))


# 3️⃣ Baseline Model (ไม่แก้ imbalance)
baseline_model = LogisticRegression(max_iter=200)
baseline_model.fit(X_train, y_train)
y_pred_baseline = baseline_model.predict(X_test)
print("\n🔹 Performance on Imbalanced Data:")
print(classification_report(y_test, y_pred_baseline))


# 4️⃣ Oversampling (SMOTE) → เพิ่ม minority class
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
print("Class distribution after SMOTE:", Counter(y_train_smote))

smote_model = LogisticRegression(max_iter=200)
smote_model.fit(X_train_smote, y_train_smote)
y_pred_smote = smote_model.predict(X_test)

print("\n🔹 Performance After SMOTE:")
print(classification_report(y_test, y_pred_smote))


# 5️⃣ Undersampling → ลด majority class
undersample = RandomUnderSampler(random_state=42)
X_train_under, y_train_under = undersample.fit_resample(X_train, y_train)
print("Class distribution after Undersampling:", Counter(y_train_under))

undersample_model = LogisticRegression(max_iter=200)
undersample_model.fit(X_train_under, y_train_under)
y_pred_under = undersample_model.predict(X_test)

print("\n🔹 Performance After Undersampling:")
print(classification_report(y_test, y_pred_under))


# 6️⃣ ใช้ Class Weights → ให้ weight class น้อยกว่าเยอะกว่า
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
weights = dict(enumerate(class_weights))

weighted_model = LogisticRegression(max_iter=200, class_weight=weights)
weighted_model.fit(X_train, y_train)
y_pred_weighted = weighted_model.predict(X_test)

print("\n🔹 Performance Using Class Weights:")
print(classification_report(y_test, y_pred_weighted))


# 7️⃣ ฟังก์ชัน Plot Confusion Matrix# -------------------------------
def plot_confusion_matrix(y_true, y_pred, title):
    conf_matrix = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5,4))
    sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues",
                xticklabels=["No Diabetes","Diabetes"],
                yticklabels=["No Diabetes","Diabetes"])
    plt.title(title)
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.show()


# Plot Confusion Matrix สำหรับแต่ละวิธี
plot_confusion_matrix(y_test, y_pred_baseline, "Baseline (Imbalanced)")
plot_confusion_matrix(y_test, y_pred_smote, "After SMOTE (Oversampling)")
plot_confusion_matrix(y_test, y_pred_under, "After Undersampling")
plot_confusion_matrix(y_test, y_pred_weighted, "Using Class Weights")
